# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:** https://github.com/Arito1/css4007-hw3

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> I used **Claude Code (Claude Opus 5.5)** as a coding assistant. It wrote the code in this notebook
> (both retrievers, the RAG pipeline, the token accounting), checked the Part A arithmetic, ran the
> notebook, and drafted the tables and written answers from the outputs printed below. The ✓/✗ marks
> in C2 were made by reading each answer against `reference`; they are recorded in a code cell so the
> counts are computed, not typed.
>
> **Provider.** The instructor allowed **Groq** instead of OpenAI. Groq's free tier allows 200k tokens
> per model per day, and HW2 used up the budget of the two gpt-oss models, so every call in Part C
> goes to **`qwen/qwen3.8-27b`** on Groq (one model for all three systems, so only what my code sends
> changes between them). Each response is stored in `llm_cache.json` the first time it is made; running
> the notebook again replays the same responses, so the analysis below matches the outputs above it.
> Delete that file to make fresh calls.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

Tokenisation (README §3): lowercase, split at spaces and hyphens, drop punctuation, no stop words removed.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** how, does, rag, retrieve, information  (N = 3 documents)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln((3 − 0 + 0.5)/(0 + 0.5)) = ln 7 = 1.9459 |
| does | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| rag | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| information | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

> No query term occurs in any document, so every term has f(t, d) = 0 and every term's contribution is
> idf(t) · 0 · (k1 + 1) / (0 + k1 · (1 − b + b·|d|/avgdl)) = 0.
>
> - D1 = 1.9459·0 + 1.9459·0 + 1.9459·0 + 1.9459·0 + 1.9459·0 = **0**
> - D2 = 0 + 0 + 0 + 0 + 0 = **0** (D2 has *retrieves* and *retrieval*, but neither is the token *retrieve*)
> - D3 = 0 + 0 + 0 + 0 + 0 = **0**
>
> No term contributes. (The length normalisation would be 1 − 0.75 + 0.75·9/8.3333 = 1.06 for D1,
> 1.15 for D2 and 0.79 for D3, but it multiplies nothing.)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 1 (tied) |
| D2 | 0 | 1 (tied) |
| D3 | 0 | 1 (tied) |

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 25 / 3 = **8.3333** (stemming changes words, not the number of tokens)

**Stemmed query terms:** how, **doe**, rag, **retriev**, **inform**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | **2** | 0 | 1 | ln((3 − 1 + 0.5)/(1 + 0.5)) = ln(2.5/1.5) = ln 1.6667 = **0.5108** |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

(*gener* now appears in D1 and D2, but it is not a query term, so it does not count.)

**BM25 of each document**, worked:

> Only *retriev* in D2 contributes.
>
> - D2, term *retriev*: f = 2, |d| = 10.
>   Denominator: f + k1·(1 − b + b·|d|/avgdl) = 2 + 1.5·(1 − 0.75 + 0.75·10/8.3333) = 2 + 1.5·(0.25 + 0.9) = 2 + 1.5·1.15 = 2 + 1.725 = **3.725**
>   Numerator: idf · f · (k1 + 1) = 0.5108 · 2 · 2.5 = **2.5541**
>   Contribution: 2.5541 / 3.725 = **0.6857**
> - D2 = 0 (how) + 0 (doe) + 0 (rag) + 0.6857 (retriev) + 0 (inform) = **0.6857**
> - D1 = 0 + 0 + 0 + 0 + 0 = **0**
> - D3 = 0 + 0 + 0 + 0 + 0 = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0 | 2 (tied with D3) |
| D2 | 0.6857 | 1 |
| D3 | 0 | 2 (tied with D1) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **(a): none.** All three documents score exactly 0, so BM25 has no first document: the order is a
> tie, and a library would just return them in index order (D1, D2, D3), which would *look* like
> "D1 first" but means nothing. It is not an arithmetic mistake. BM25 only rewards exact shared tokens,
> and this query shares none with any document: *retrieve* ≠ *retrieves* ≠ *retrieval*; *RAG* is the
> acronym of *Retrieval-Augmented Generation* but BM25 sees three unrelated strings; *information*,
> *how* and *does* appear nowhere. The document that is obviously about the query (D2) gets nothing.
>
> **(b): D2**, with 0.6857, and D1 and D3 stay tied at 0. Stemming maps *retrieve*, *retrieves* and
> *retrieval* to the same token *retriev*, which occurs twice in D2. Its idf is low (0.5108, because it
> is in one of only three documents) and D2 is slightly longer than average (10 vs 8.33), which
> lowers the score a little, but it is the only match there is. *RAG* still matches nothing: stemming
> fixes word endings, not acronyms.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = **0.96437**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = **0.78** | 0.96437 | √(0.49 + 0.01 + 0.16) = √0.66 = 0.81240 | 0.78 / (0.96437·0.81240) = 0.78 / 0.78345 = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = **1.08** | 0.96437 | √(0.81 + 0.09 + 0.36) = √1.26 = 1.12250 | 1.08 / (0.96437·1.12250) = 1.08 / 1.08250 = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = **0.34** | 0.96437 | √(0.01 + 0.64 + 0.04) = √0.69 = 0.83066 | 0.34 / (0.96437·0.83066) = 0.34 / 0.80107 = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (0.9977) — but only just: D1 is 0.9956, a difference of 0.002. D3 (0.4244) is far behind.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> Not with (a): BM25 as written gives all three documents 0 and ranks nothing, while the embedding
> ranking is D2 > D1 > D3. With (b) the top agrees (D2 first), but BM25 cannot separate D1 from D3
> (both 0), whereas the embeddings put D1 almost level with D2 and D3 clearly last.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 compares strings; embeddings compare directions in a meaning space. The query says *RAG* and
> D2 says *Retrieval-Augmented Generation*: the same thing, zero shared tokens, so BM25 gives 0 and only
> rescues D2 through a stemmed *retrieve → retriev*. The query asks about *information*, and D1 talks
> about models that *answer questions*, D2 about *relevant documents*: related meanings, no shared word.
> An embedding model places "how does RAG retrieve information" close to D2 and also close to D1
> (both about language models answering) even though D1 shares no token with the query, and far from D3,
> which is about vectors. So BM25 fails when the right document uses different words (acronym,
> inflection, synonym), and embeddings can rank a document high because its topic is near even when
> it does not contain the answer (D1 is 0.9956 and says nothing about retrieval).

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
# Check of the Part A arithmetic. The working is written out by hand above; this only confirms it.
import math, re
import numpy as np

docs = {"D1": "Large language models can generate text and answer questions.",
        "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
        "D3": "Embeddings represent text as numerical vectors."}
query = "How does RAG retrieve information?"
STEMS = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
         "generate": "gener", "generating": "gener", "generation": "gener",
         "information": "inform", "does": "doe"}

def tok(s):  # lowercase, split at spaces and hyphens, drop punctuation
    return [t for t in re.split(r"[ \-]+", re.sub(r"[^\w\s-]", "", s.lower())) if t]

def bm25_by_hand(stem, k1=1.5, b=0.75):
    D = {k: [STEMS.get(t, t) if stem else t for t in tok(v)] for k, v in docs.items()}
    q = [STEMS.get(t, t) if stem else t for t in tok(query)]
    N, avgdl = len(D), sum(map(len, D.values())) / len(D)
    print(("(b) stemmed" if stem else "(a) as written"), "| avgdl =", round(avgdl, 4), "| query:", q)
    for t in q:
        n = sum(t in d for d in D.values())
        print(f"   {t:12s} tf={[d.count(t) for d in D.values()]} n={n} idf={math.log((N - n + 0.5) / (n + 0.5)):.4f}")
    for k, d in D.items():
        score = sum(math.log((N - sum(t in x for x in D.values()) + 0.5) / (sum(t in x for x in D.values()) + 0.5))
                    * d.count(t) * (k1 + 1) / (d.count(t) + k1 * (1 - b + b * len(d) / avgdl)) for t in q)
        print(f"   {k}: |d|={len(d):2d}  BM25={score:.4f}")

bm25_by_hand(stem=False)
bm25_by_hand(stem=True)

Q = np.array([0.8, 0.2, 0.5])
print("\nA2  |Q| =", round(float(np.linalg.norm(Q)), 5))
for k, v in {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}.items():
    v = np.array(v)
    print(f"   {k}: dot={Q @ v:.2f}  |D|={np.linalg.norm(v):.5f}  cosine={Q @ v / np.linalg.norm(Q) / np.linalg.norm(v):.4f}")

(a) as written | avgdl = 8.3333 | query: ['how', 'does', 'rag', 'retrieve', 'information']
   how          tf=[0, 0, 0] n=0 idf=1.9459
   does         tf=[0, 0, 0] n=0 idf=1.9459
   rag          tf=[0, 0, 0] n=0 idf=1.9459
   retrieve     tf=[0, 0, 0] n=0 idf=1.9459
   information  tf=[0, 0, 0] n=0 idf=1.9459
   D1: |d|= 9  BM25=0.0000
   D2: |d|=10  BM25=0.0000
   D3: |d|= 6  BM25=0.0000
(b) stemmed | avgdl = 8.3333 | query: ['how', 'doe', 'rag', 'retriev', 'inform']
   how          tf=[0, 0, 0] n=0 idf=1.9459
   doe          tf=[0, 0, 0] n=0 idf=1.9459
   rag          tf=[0, 0, 0] n=0 idf=1.9459
   retriev      tf=[0, 2, 0] n=1 idf=0.5108
   inform       tf=[0, 0, 0] n=0 idf=1.9459
   D1: |d|= 9  BM25=0.0000
   D2: |d|=10  BM25=0.6857
   D3: |d|= 6  BM25=0.0000

A2  |Q| = 0.96437
   D1: dot=0.78  |D|=0.81240  cosine=0.9956
   D2: dot=1.08  |D|=1.12250  cosine=0.9977
   D3: dot=0.34  |D|=0.83066  cosine=0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json, os, re, time, hashlib, warnings
from pathlib import Path
import numpy as np
from dotenv import load_dotenv

load_dotenv()                                   # OPENAI/GROQ key and HF_TOKEN come from .env, never from a cell
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
warnings.filterwarnings("ignore")

DATA = Path("data")
corpus = [json.loads(line) for line in open(DATA / "corpus.jsonl", encoding="utf-8")]
queries = json.load(open(DATA / "queries.json", encoding="utf-8"))["queries"]
qa = json.load(open(DATA / "qa.json", encoding="utf-8"))["questions"]

ids = [c["id"] for c in corpus]
by_id = {c["id"]: c for c in corpus}
print(f"{len(corpus)} chunks: {sum(c['source'] == 'wikipedia' for c in corpus)} Wikipedia (KZ-), "
      f"{sum(c['source'] == 'grant_office_handbook' for c in corpus)} handbook (GO-)")
print("languages:", {l: sum(c["lang"] == l for c in corpus) for l in sorted({c["lang"] for c in corpus})})
print(f"{len(queries)} fixed queries, {len(qa)} QA questions")

74 chunks: 51 Wikipedia (KZ-), 23 handbook (GO-)
languages: {'en': 71, 'kk': 1, 'ru': 2}
8 fixed queries, 6 QA questions


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

> Lowercase, then `re.findall(r"\w+")`: a token is a run of Unicode letters/digits, so Kazakh and
> Russian words survive whole and punctuation and hyphens split tokens (`GR-305` → `gr`, `305`;
> `Nur-Sultan` → `nur`, `sultan`). **No stemming and no stop-word removal**, in any language. Index
> over the chunk `text` only (not the title), `BM25Okapi(k1=1.5, b=0.75)`. Note that `rank_bm25`'s
> idf is the Part A formula, except that a negative idf (a term in more than half the chunks, like
> *the*) is replaced by a small positive floor.

In [3]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(c["text"]) for c in corpus], k1=1.5, b=0.75)

def bm25_top(q, k=3):
    scores = bm25.get_scores(tokenize(q))
    order = np.argsort(-scores, kind="stable")[:k]
    return [(ids[i], float(scores[i])) for i in order]

print("index built over", len(bm25.doc_len), "chunks; avgdl =", round(bm25.avgdl, 1), "tokens")
print("example:", tokenize("What is form GR-305 used for?"), "->", bm25_top("What is form GR-305 used for?"))

index built over 74 chunks; avgdl = 93.8 tokens
example: ['what', 'is', 'form', 'gr', '305', 'used', 'for'] -> [('GO-08', 15.825491699499644), ('GO-12', 8.195256931904641), ('GO-06', 8.07003113015204)]


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
emb_model = SentenceTransformer("google/embeddinggemma-300m", token=os.environ.get("HF_TOKEN"))  # float32 (default)
doc_emb = emb_model.encode_document([c["text"] for c in corpus], show_progress_bar=False)
print(f"model loaded and {len(corpus)} chunks embedded in {time.time() - t0:.1f}s on CPU;",
      f"matrix {doc_emb.shape}, dtype {doc_emb.dtype}")

def emb_top(q, k=3):
    q_emb = emb_model.encode_query([q], show_progress_bar=False)
    scores = emb_model.similarity(q_emb, doc_emb)[0].numpy()        # cosine similarity to every chunk
    order = np.argsort(-scores, kind="stable")[:k]
    return [(ids[i], float(scores[i])) for i in order]

print("example:", emb_top("What is form GR-305 used for?"))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:   9%|▉         | 28/314 [00:00<00:01, 270.71it/s]

Loading weights:  18%|█▊        | 56/314 [00:00<00:01, 177.33it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1023.63it/s]

model loaded and 74 chunks embedded in 23.8s on CPU; matrix (74, 768), dtype float32
example: [('GO-08', 0.6955163478851318), ('GO-12', 0.4608723819255829), ('GO-23', 0.4504862129688263)]


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What is form GR-210? | GO-07, GO-03 | **BM25.** The answer hinges on one exact code. BM25 matches the tokens `gr` + `210` literally, and `210` occurs only in the GR-210 chunks. The embedding model sees three near-identical form names (GR-104, GR-210, GR-305) and the word *form* everywhere, so I expected it to blur them. |
| Q-10 | Какой минимальный средний балл нужен, чтобы получить грант? | GO-02 | **Embeddings.** The question is in Russian; the rule (GPA ≥ 2.67) is stated only in English, in GO-02. BM25 shares no token with any English chunk, and the only Russian chunks (GO-15 payments, GO-23 appeals) do not answer it. EmbeddingGemma is multilingual. (Risk: the archived GO-05 also talks about a minimum GPA.) |
| Q-11 | GR-104 | GO-06 | **BM25.** Added after Q-09's prediction failed (both retrievers found GR-210), with `relevant` fixed before running: a bare form code, as a student would paste it, with no other words to carry meaning. GO-06 is the chunk that defines GR-104; five chunks mention it. |
| Q-12 | What happened 456 times? | KZ-38 | **BM25.** Added for the same reason: the only content word is a rare number. `456` occurs in exactly one chunk (KZ-38, "456 nuclear tests"); "what happened … times" means almost nothing to an embedding. |
| Q-13 | 456 | KZ-38 | **BM25.** Added together with Q-12's diagnosis (its question words outweighed the number): the number alone. `456` is in exactly one chunk, so BM25 has one non-zero score; an embedding of a bare number has almost no meaning to place. |

In [5]:
# Written before running anything below; `relevant` decided from reading the corpus.
# Q-11 and Q-12 were added after the first run, when Q-09 did not separate the retrievers;
# their `relevant` was fixed before they were run.
my_queries = [
    {"id": "Q-09", "text": "What is form GR-210?", "relevant": ["GO-07", "GO-03"],
     "expect": "BM25 (exact form code)"},
    {"id": "Q-10", "text": "Какой минимальный средний балл нужен, чтобы получить грант?", "relevant": ["GO-02"],
     "expect": "embeddings (Russian question, English answer)"},
    {"id": "Q-11", "text": "GR-104", "relevant": ["GO-06"],
     "expect": "BM25 (a bare code, no other words)"},
    {"id": "Q-12", "text": "What happened 456 times?", "relevant": ["KZ-38"],
     "expect": "BM25 (one rare number)"},
    {"id": "Q-13", "text": "456", "relevant": ["KZ-38"],
     "expect": "BM25 (the rare number alone)"},
]
all_queries = queries + my_queries
print(len(all_queries), "queries:", [q["id"] for q in all_queries])

13 queries: ['Q-01', 'Q-02', 'Q-03', 'Q-04', 'Q-05', 'Q-06', 'Q-07', 'Q-08', 'Q-09', 'Q-10', 'Q-11', 'Q-12', 'Q-13']


## Results for all ten or more queries: top 3 from each retriever

In [6]:
def first_line(cid, n=70):
    c = by_id[cid]
    return f"{c['title'][:38]:38s} | {c['text'][:n]}…"

results = {}
for q in all_queries:
    b, e = bm25_top(q["text"]), emb_top(q["text"])
    hit_b = any(cid in q["relevant"] for cid, _ in b)
    hit_e = any(cid in q["relevant"] for cid, _ in e)
    results[q["id"]] = {"bm25": b, "emb": e, "hit_bm25": hit_b, "hit_emb": hit_e}
    print(f"\n{q['id']}: {q['text']}    relevant = {q['relevant']}")
    for name, top, hit in (("BM25", b, hit_b), ("EmbeddingGemma", e, hit_e)):
        print(f"  {name}  hit@3 = {int(hit)}")
        for r, (cid, s) in enumerate(top, 1):
            mark = "*" if cid in q["relevant"] else " "
            print(f"    {r}. {mark}{cid:6s} {s:7.3f}  {first_line(cid)}")

print("\nhit@3 totals:  BM25", sum(r["hit_bm25"] for r in results.values()), f"/{len(results)}",
      "   EmbeddingGemma", sum(r["hit_emb"] for r in results.values()), f"/{len(results)}")
print("(* = a chunk listed in `relevant`)")


Q-01: What is form GR-305 used for?    relevant = ['GO-08', 'GO-12', 'GO-23']
  BM25  hit@3 = 1
    1. *GO-08   15.825  Form GR-305: appeals                   | Form GR-305 is used to appeal a decision of the Grant Office. It is th…
    2. *GO-12    8.195  How to appeal a decision               | A student who disagrees with a decision may appeal within 14 calendar …
    3.  GO-06    8.070  Form GR-104: the application form      | Form GR-104 is the application form for the need-based study grant. It…
  EmbeddingGemma  hit@3 = 1
    1. *GO-08    0.696  Form GR-305: appeals                   | Form GR-305 is used to appeal a decision of the Grant Office. It is th…
    2. *GO-12    0.461  How to appeal a decision               | A student who disagrees with a decision may appeal within 14 calendar …
    3. *GO-23    0.450  Апелляция: кратко                      | Если студент не согласен с решением, он может подать апелляцию по форм…



Q-02: My family is poor. Is there financial help for students like me?    relevant = ['GO-01', 'GO-02', 'GO-03']
  BM25  hit@3 = 0
    1.  KZ-17    8.726  Lake Balkhash (part 2)                 | The lake currently covers about 16,400 km2 (6,300 sq mi). However, lik…
    2.  GO-20    8.311  Who cannot apply                       | The need-based grant is for full-time undergraduate students only. Par…
    3.  KZ-25    7.511  Beshbarmak (part 2)                    | Beshbarmak is usually made from finely chopped boiled meat, mixed with…
  EmbeddingGemma  hit@3 = 1
    1. *GO-01    0.445  About the need-based study grant 2026  | The need-based study grant 2026 is run by the Grant Office for full-ti…
    2.  GO-10    0.412  What counts as proof of income         | Proof of income must accompany form GR-210. Accepted: a salary certifi…
    3.  GO-20    0.409  Who cannot apply                       | The need-based grant is for full-time undergraduate students only. Par…



Q-03: Өтінішті қашанға дейін тапсыру керек?    relevant = ['GO-09', 'GO-14']
  BM25  hit@3 = 1
    1. *GO-14   10.698  Өтініш беру мерзімі және тәртібі       | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту…
    2.  KZ-01    0.000  Kazakhstan (part 1)                    | Kazakhstan, officially the Republic of Kazakhstan, is a landlocked cou…
    3.  KZ-02    0.000  Kazakhstan (part 2)                    | Kazakhstan is the world's ninth-largest country by land area and the l…
  EmbeddingGemma  hit@3 = 1
    1. *GO-14    0.399  Өтініш беру мерзімі және тәртібі       | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту…
    2. *GO-09    0.350  Application deadline                   | Applications for the 2026 grant open on 1 September 2026 and close on …
    3.  GO-11    0.301  When a document is missing             | If a required document is missing, the Grant Office does not refuse th…



Q-04: What is the minimum GPA for the grant?    relevant = ['GO-02']
  BM25  hit@3 = 0
    1.  GO-05   12.780  Archived: the 2025 scheme (no longer i | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum …
    2.  GO-17   10.518  Academic progress during the year      | The grant continues only while the student keeps up their studies. If …
    3.  GO-19   10.136  Priority when the budget runs short    | The grant budget is fixed each year. If there are more qualifying appl…
  EmbeddingGemma  hit@3 = 1
    1. *GO-02    0.665  Who qualifies                          | An applicant qualifies for the 2026 grant when all three conditions ho…
    2.  GO-05    0.621  Archived: the 2025 scheme (no longer i | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum …
    3.  GO-17    0.613  Academic progress during the year      | The grant continues only while the student keeps up their studies. If …



Q-05: When was Astana renamed Nur-Sultan, and why?    relevant = ['KZ-05']
  BM25  hit@3 = 1
    1. *KZ-05   15.731  Astana                                 | Astana is the capital city of Kazakhstan. With a population of 1,7 mil…
    2.  KZ-06    6.443  Almaty (part 1)                        | Almaty is the most populous city in Kazakhstan, with more than two mil…
    3.  KZ-11    5.866  Baikonur Cosmodrome (part 2)           | In 1955, the Soviet Ministry of Defense issued a decree and founded th…
  EmbeddingGemma  hit@3 = 1
    1. *KZ-05    0.651  Astana                                 | Astana is the capital city of Kazakhstan. With a population of 1,7 mil…
    2.  KZ-06    0.507  Almaty (part 1)                        | Almaty is the most populous city in Kazakhstan, with more than two mil…
    3.  KZ-03    0.419  Kazakhstan (part 3)                    | Kazakhstan has been inhabited since the Paleolithic era. In antiquity,…


...


Q-08: If my grades drop in the autumn, do I still get paid in February?    relevant = ['GO-17']
  BM25  hit@3 = 1
    1. *GO-17   17.168  Academic progress during the year      | The grant continues only while the student keeps up their studies. If …
    2.  GO-05   12.773  Archived: the 2025 scheme (no longer i | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum …
    3.  GO-04   11.932  How much the grant pays                | The grant amount depends only on the income band, not on the GPA. A st…
  EmbeddingGemma  hit@3 = 1
    1. *GO-17    0.559  Academic progress during the year      | The grant continues only while the student keeps up their studies. If …
    2.  GO-18    0.459  Academic leave and transfers           | A student who goes on academic leave keeps their grant decision, but p…
    3.  GO-15    0.453  Выплата гранта                         | Грант выплачивается двумя равными частями — в октябре и в феврале — на…



Q-09: What is form GR-210?    relevant = ['GO-07', 'GO-03']
  BM25  hit@3 = 1
    1. *GO-07   10.622  Form GR-210: the income declaration    | Form GR-210 is the household income declaration. It lists every adult …
    2.  GO-13    9.521  Changes in circumstances               | A student whose household income changes during the year must report i…
    3.  GO-10    8.587  What counts as proof of income         | Proof of income must accompany form GR-210. Accepted: a salary certifi…
  EmbeddingGemma  hit@3 = 1
    1. *GO-07    0.656  Form GR-210: the income declaration    | Form GR-210 is the household income declaration. It lists every adult …
    2.  GO-10    0.511  What counts as proof of income         | Proof of income must accompany form GR-210. Accepted: a salary certifi…
    3.  GO-13    0.464  Changes in circumstances               | A student whose household income changes during the year must report i…



Q-10: Какой минимальный средний балл нужен, чтобы получить грант?    relevant = ['GO-02']
  BM25  hit@3 = 0
    1.  GO-14    4.628  Өтініш беру мерзімі және тәртібі       | 2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін сту…
    2.  GO-15    4.211  Выплата гранта                         | Грант выплачивается двумя равными частями — в октябре и в феврале — на…
    3.  KZ-01    0.000  Kazakhstan (part 1)                    | Kazakhstan, officially the Republic of Kazakhstan, is a landlocked cou…
  EmbeddingGemma  hit@3 = 1
    1. *GO-02    0.561  Who qualifies                          | An applicant qualifies for the 2026 grant when all three conditions ho…
    2.  GO-19    0.510  Priority when the budget runs short    | The grant budget is fixed each year. If there are more qualifying appl…
    3.  GO-05    0.505  Archived: the 2025 scheme (no longer i | ARCHIVED — kept for reference only. Under the 2025 scheme the minimum …

Q-11: GR-104    relevant = ['GO-06']
  B


Q-12: What happened 456 times?    relevant = ['KZ-38']
  BM25  hit@3 = 1
    1.  GO-13    4.490  Changes in circumstances               | A student whose household income changes during the year must report i…
    2.  KZ-47    3.197  Caspian Sea (part 3)                   | The Caspian Sea is five times as big as Lake Superior and is home to a…
    3. *KZ-38    2.869  Semipalatinsk Test Site (part 2)       | The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 194…
  EmbeddingGemma  hit@3 = 1
    1. *KZ-38    0.215  Semipalatinsk Test Site (part 2)       | The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 194…
    2.  GO-07    0.153  Form GR-210: the income declaration    | Form GR-210 is the household income declaration. It lists every adult …
    3.  GO-08    0.132  Form GR-305: appeals                   | Form GR-305 is used to appeal a decision of the Grant Office. It is th…



Q-13: 456    relevant = ['KZ-38']
  BM25  hit@3 = 1
    1. *KZ-38    2.869  Semipalatinsk Test Site (part 2)       | The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 194…
    2.  KZ-01    0.000  Kazakhstan (part 1)                    | Kazakhstan, officially the Republic of Kazakhstan, is a landlocked cou…
    3.  KZ-02    0.000  Kazakhstan (part 2)                    | Kazakhstan is the world's ninth-largest country by land area and the l…
  EmbeddingGemma  hit@3 = 0
    1.  GO-07    0.194  Form GR-210: the income declaration    | Form GR-210 is the household income declaration. It lists every adult …
    2.  GO-09    0.163  Application deadline                   | Applications for the 2026 grant open on 1 September 2026 and close on …
    3.  GO-08    0.146  Form GR-305: appeals                   | Form GR-305 is used to appeal a decision of the Grant Office. It is th…

hit@3 totals:  BM25 9 /13    EmbeddingGemma 12 /13
(* = a chunk listed in `relevant`)


## B3 — comparison

From the output above (top 3 in rank order; ✓ = a `relevant` chunk is in the top 3). Q-11 to Q-13 are
my extra queries.

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-13, GO-10 | ✓ | GO-07, GO-10, GO-13 | ✓ |
| Q-10 | GO-14, GO-15, KZ-01 | ✗ | GO-02, GO-19, GO-05 | ✓ |
| **hit@3 total (Q-01 … Q-10)** | | 6/10 | | 10/10 |
| Q-11 | GO-06, GO-14, GO-15 | ✓ | GO-06, GO-14, GO-09 | ✓ |
| Q-12 | GO-13, KZ-47, KZ-38 | ✓ | KZ-38, GO-07, GO-08 | ✓ |
| Q-13 | KZ-38, KZ-01, KZ-02 | ✓ | GO-07, GO-09, GO-08 | ✗ |
| **hit@3 total (all 13)** | | 9/13 | | 12/13 |

My predictions: Q-10 (embeddings) was right. Q-09 and Q-11 (BM25) were wrong — both retrievers put the
right form first. Q-12 was a tie on hit@3 but embeddings ranked KZ-38 first and BM25 third. Only Q-13
went to BM25.

**Both retrievers agree:** which query, and why was it easy?

> **Q-06** ("Where did the Soviet Union test its nuclear weapons?"): both put KZ-37 and KZ-38 — the two
> Semipalatinsk chunks — in places 1 and 2 (BM25 19.09 and 17.17; cosine 0.596 and 0.510). It was easy
> because the lexical and the semantic signal point at the same text: the answer chunks literally
> contain *Soviet Union*, *nuclear* and *test(s)*, those words are rare elsewhere in the corpus (high
> idf), and the topic is unlike anything else in it. Q-05 (Astana) is easy for the same reason.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> **Q-13** ("456"). BM25 matched the exact token **`456`**, which occurs in one chunk only (KZ-38, "The
> Soviet Union conducted 456 nuclear tests…"), so KZ-38 got the only non-zero score in the corpus
> (2.869; places 2 and 3 are 0.000). The embedding model blurred the bare number into "a chunk with
> numbers and codes in it" and returned GO-07 (GR-210), GO-09 (dates) and GO-08 (GR-305) at cosine
> 0.15–0.19 — noise level — missing KZ-38 entirely. Two notes from my own runs: the form codes I expected
> it to blur (Q-09 *GR-210*, Q-11 *GR-104*) it did **not** blur, because each code is spelled out in
> the first sentence of its chunk; and BM25's win is fragile — in Q-12 ("What happened 456 times?") the
> same `456` contributed only 2.869 because KZ-38 is long (168 tokens vs. avgdl 93.8, so length
> normalisation cuts it), and the word *what* alone gave GO-13 4.490 and first place.

**Embeddings did better:** which query, and what could BM25 not match?

> **Q-07** ("Can the grant be sent to my mother's card?"). The answer is GO-15, which is in **Russian**:
> "перевод на счёт родителей или других лиц невозможен". BM25 shares no token with it (every query
> term contributes 0), so it ranked GO-21 first on *can* (3.95) and *sent* (4.39) — "questions can also
> be sent through the portal" — an answer to a different question. EmbeddingGemma matched the meaning
> across languages (*mother's card* ↔ *счёт родителей*) and put GO-15 first. Q-10 (Russian query,
> English answer) and Q-02 (*poor* vs *low-income household*, a paraphrase) fail BM25 the same way.
> **Q-04** is the instructive one: BM25's first hit is **GO-05, titled "Archived: the 2025 scheme (no
> longer in force)"**, because it contains the exact word *minimum* (+4.57) — "the minimum GPA for the
> grant was 3.0" — while the current rule GO-02 says "at least 2.67" and has no *minimum* at all.
> Embeddings put GO-02 first, but GO-05 second at 0.621 vs 0.665: they nearly fell for it too.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

> **Model actually used:** `qwen/qwen3.8-27b` on Groq, for every call (see the disclosure at the top).

In [7]:
from openai import OpenAI
import tiktoken

MODEL = "qwen/qwen3.8-27b"
client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1", max_retries=8)
enc = tiktoken.get_encoding("o200k_base")

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

CACHE_FILE = Path("llm_cache.json")
cache = json.loads(CACHE_FILE.read_text(encoding="utf-8")) if CACHE_FILE.exists() else {}

def ask(messages):
    """One call. Replayed from llm_cache.json if this exact (model, messages) was sent before."""
    key = hashlib.sha256(json.dumps([MODEL, messages], ensure_ascii=False).encode()).hexdigest()
    if key not in cache:
        resp = client.chat.completions.create(model=MODEL, messages=messages, max_completion_tokens=512)
        cache[key] = {"answer": resp.choices[0].message.content, "finish_reason": resp.choices[0].finish_reason,
                      "prompt_tokens": resp.usage.prompt_tokens,
                      "completion_tokens": resp.usage.completion_tokens}
        CACHE_FILE.write_text(json.dumps(cache, ensure_ascii=False, indent=1), encoding="utf-8")
    return cache[key]

def build_context(hits):
    return "\n\n".join(f"[{i}] {by_id[cid]['title']}\n{by_id[cid]['text']}" for i, (cid, _) in enumerate(hits, 1))

def build_messages(question, hits):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n\n{build_context(hits)}\n\nQuestion: {question}"}]

RETRIEVERS = {"bm25": bm25_top, "emb": emb_top}

def rag(question, retriever, k=3):
    hits = RETRIEVERS[retriever](question, k)
    return hits, ask(build_messages(question, hits))

def llm_only(question):
    return ask([{"role": "user", "content": question}])     # the question and nothing else

print("model:", MODEL, "| cached responses on disk:", len(cache))

model: qwen/qwen3.8-27b | cached responses on disk: 22


## C1 — the complete prompt, printed before it is sent

In [8]:
c1 = next(q for q in qa if q["id"] == "C-01")
hits = emb_top(c1["question"], 3)
messages = build_messages(c1["question"], hits)

print("=" * 30, "THE COMPLETE PROMPT, EXACTLY AS SENT", "=" * 30)
for m in messages:
    print(f"----- role: {m['role']} -----")
    print(m["content"])
print("=" * 96)

context = build_context(hits)
n_system, n_question, n_context = (len(enc.encode(x)) for x in (SYSTEM, c1["question"], context))
n_user_message = len(enc.encode(messages[1]["content"]))
n_total = n_system + n_user_message
print(f"\nSystem instructions : {n_system:4d} tokens")
print(f"Question            : {n_question:4d} tokens")
print(f"Retrieved context   : {n_context:4d} tokens")
print(f"Framing ('Context:', 'Question:', blank lines): {n_user_message - n_question - n_context} tokens")
print(f"Total input (system + whole user message, tiktoken o200k_base): {n_total} tokens")

out = ask(messages)
print(f"usage.prompt_tokens from the response: {out['prompt_tokens']}   (difference {out['prompt_tokens'] - n_total:+d})")
print(f"\nChunks retrieved: {len(hits)}  -> {[cid for cid, _ in hits]}")
for cid, s in hits:
    c = by_id[cid]
    print(f"  {cid}: {len(enc.encode(c['text']))} tokens, {len(c['text'])} characters (cosine {s:.3f})")
print(f"Context block as sent: {n_context} tokens, {len(context)} characters")
print("\nAnswer:", out["answer"])

============================== THE COMPLETE PROMPT, EXACTLY AS SENT ==============================
----- role: system -----
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].
----- role: user -----
Context:

[1] How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 ten

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 18 |
| Retrieved context | 302 |
| (framing: "Context:", "Question:", blank lines) | 4 |
| **Total input** | **368** |
| **`usage.prompt_tokens` from the response** | **415** (+47) |

- Number of retrieved chunks: **3** (EmbeddingGemma top 3 for C-01: GO-04, GO-03, GO-15)
- Tokens per chunk: GO-04 **90**, GO-03 **104**, GO-15 **85** (279 for the three texts; the other 23 of
  the 302 are the block numbers `[1]`–`[3]` and the titles)
- Total context size (tokens / characters): **302 tokens / 1,197 characters**

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> Two reasons. **(1) The chat template.** The API does not send my strings as they are: each message is
> wrapped in role markers and special tokens, and the prompt ends with the start of the assistant's
> turn. I measured this on the six LLM-only calls in C2, which send one user message: the provider
> counted exactly **12 tokens more** than tiktoken every time (C-01 18 → 30, C-02 21 → 33, … C-06 15 → 28).
> With a system message and a user message the wrapper is roughly 15–20 tokens. **(2) A different
> tokenizer.** `o200k_base` is OpenAI's tokenizer; the call went to Qwen on Groq, which has its own BPE
> vocabulary, so the same text splits into a different number of pieces — most of all the Russian
> chunk GO-15. The remaining ~30 tokens of the +47 come from that. (In HW1 the gap was only the
> template, because tokenizer and model matched; here both effects add up.)

**What exactly does retrieval add to the model's input?**

> Literally: **302 tokens of text** — three numbered blocks, each "`[n] <chunk title>`" followed by the
> chunk's text, pasted into the user message between "Context:" and "Question:". That is 82% of what
> was sent (302 of 368). Nothing else: no file, no link, no index, no access to the other 71 chunks,
> no score. The model receives a longer prompt and continues it; anything it gets right about the
> handbook has to be in those 1,197 characters.

## C2 — three systems, six questions

In [9]:
c2 = {}
for q in qa:
    row = {"llm": llm_only(q["question"])["answer"]}
    for r in ("bm25", "emb"):
        hits, out = rag(q["question"], r)
        row[r] = out["answer"]
        row[r + "_hits"] = [cid for cid, _ in hits]
    c2[q["id"]] = row
    print("=" * 100)
    print(f"{q['id']}: {q['question']}")
    print(f"reference: {q['reference']}   (relevant: {q['relevant']})")
    print(f"\n[LLM only]\n{row['llm']}")
    for r, name in (("bm25", "BM25 RAG"), ("emb", "Embedding RAG")):
        got = [cid for cid in row[r + '_hits'] if cid in q["relevant"]]
        print(f"\n[{name}]  context = {row[r + '_hits']}   relevant chunk in context: {got or 'NONE'}\n{row[r]}")

C-01: How much does a student in income band 2 receive, and when is it paid?
reference: 150,000 tenge for the academic year, in two equal instalments: October and February.   (relevant: ['GO-04', 'GO-15'])

[LLM only]
Based on standard government student grant structures (typically referring to Australia’s **Government Assistance for Overseas Students** or similar means-tested schemes, as "Income Band 2" is most commonly associated with Australian Centerlink grants or overseas student subsidies), here is the general information.

**Note:** Policies vary significantly by country and year. The most common reference for "Income Band 2" in student contexts is **Australia’s Student Assistance Measures**. However, if you are referring to a different country (e.g., UK, Canada, New Zealand), please specify.

### For Australian Overseas Students (Government Assistance):
As of recent financial years (e.g., 2023–2024), the Government Assistance for Overseas Students (SAOS) is not typically funded

C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
reference: 14 calendar days from the date on the decision letter, using form GR-305.   (relevant: ['GO-08', 'GO-12', 'GO-23'])

[LLM only]
I am an AI assistant and do not have access to your specific grant details, funding body, or location. Therefore, I **cannot** provide a definitive answer about your specific appeal deadline or form. Grant appeal processes vary significantly depending on:

1. **The granting organization** (e.g., government agency, private foundation, university, corporate program).
2. **Jurisdiction** (country/state/province regulations).
3. **Type of grant** (federal, state, international, institutional).

### What You Should Do Immediately:

1. **Check Your Rejection Letter/Email**:  
   - Most funders include appeal instructions, including the **deadline** (e.g., “within 30 days of the decision date”) and the **required form or portal** for appeals, directly in the deci

C-03: What is the minimum GPA needed to qualify for the need-based grant?
reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)   (relevant: ['GO-02'])

[LLM only]
Because you haven't specified which university, scholarship organization, or funding program you are referring to, **I cannot provide a specific GPA number.**

"Need-based grants" are administered by individual institutions (colleges, universities, governments, or private foundations), and each has its **own specific eligibility requirements**.

### Key Points to Consider:

1. **Many need-based grants do NOT have a minimum GPA** at the point of application. Accessibility is often based primarily on financial need (as determined by the FAFSA, CSS Profile, or institutional forms) rather than academic standing.
2. **Some institutions or external scholarships** may require a minimum GPA (e.g., 2.0, 2.5, or 3.0 on a 4.0 scale) to maintain eligibility or to receive continued f

C-04: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
reference: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.   (relevant: ['KZ-43'])

[LLM only]
The decision to switch the Kazakh alphabet from Cyrillic to the Latin script was formally announced by President Kassym-Jomart Tokayev in **September 2018**.

Specifically:
- **Announcement Date:** On **September 29, 2018**, President Tokayev declared the government's plan to introduce a new alphabet based on the Latin script.
- **Legal Framework:** In **2017**, a law had already been enacted to establish the Latin-based Kazakh alphabet, providing the legal foundation for the transition.
- **Transition Timeline:** The original plan called for the switch to be completed by **2025**. However, this timeline has been subject to delays and debates.

As of the most recent official statements (up to 2023–2024

C-06: I am a master's student. How much need-based grant can I get?
reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.   (relevant: ['GO-20'])

[LLM only]
The specific amount of need-based financial aid you can receive as a master’s student varies significantly depending on several factors, including your country, university policies, citizenship status, field of study, and how you define "need-based" vs. "merit-based" or "employment-based" funds.

Here’s a breakdown to help you understand what to expect:

### 1. **Clarify "Need-Based" in Graduate School**
Unlike undergraduate education, **true need-based grants are rare for master’s students** in the U.S. and many other countries. Most graduate funding consists of:
- **Assistantships** (TA/RA): You work for the university (teaching or research) in exchange for a stipend + tuition waiver.
- **Merit-Based Scholarships**: Awarded based on academic achievement, not 

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ generic "Income Band 2" schemes, e.g. "around $1,500 AUD per year" (cut off at 512 tokens) | ✓ 150,000 tenge; two equal instalments, October and February [1] | ✓ same, [1] |
| C-02 | ✗ no answer: "deadlines commonly range from 14 to 60 days", check your letter | ✓ 14 calendar days from the decision letter; form GR-305 [2][3] | ✓ 14 calendar days; GR-305 [1][2] |
| C-03 | ✗ "I cannot provide a specific GPA number"; Pell grants have no minimum (cut off) | ✓ 2.67 on the 4.0 scale [2] | ✓ 2.67 [1], and notes the archived 3.0 no longer applies [3] |
| C-04 | ✗ "announced by President Tokayev in September 2018", target "remains 2025" | ✓ October 2017 decree; "completed by 2023 to 2031", final target 2031 (muddled, see C3) | ✓ October 2017; 2025 pushed back to 2031 in January 2021 |
| C-05 | ✗ invented: founded 2013 as "International University of Tashkent" | ✓ 1963, Almaty Institute of National Economy [1] | ✓ same [1] |
| C-06 | ✗ US graduate funding: TA/RA stipends "$20,000–$40,000/year" (cut off) | ✓ "I do not know"; master's students have a separate scheme the handbook does not describe [3] | ✓ "I do not know", same reason [1] |
| **agrees with `reference`** | **0/6** | **6/6** | **6/6** |

The counts are computed from my marks in the next cell.

In [10]:

# My marks for C2, made by reading each answer above against `reference` (✓ = True).
MARKS = {
    "C-01": {"llm": False, "bm25": True,  "emb": True},
    "C-02": {"llm": False, "bm25": True,  "emb": True},
    "C-03": {"llm": False, "bm25": True,  "emb": True},
    "C-04": {"llm": False, "bm25": True,  "emb": True},
    "C-05": {"llm": False, "bm25": True,  "emb": True},
    "C-06": {"llm": False, "bm25": True,  "emb": True},
}
for system, name in (("llm", "LLM only"), ("bm25", "BM25 RAG"), ("emb", "Embedding RAG")):
    print(f"{name:14s} agrees with reference: {sum(m[system] for m in MARKS.values())}/6")
# Was a relevant chunk in the top 3 at all? (retrieval, separate from whether the answer was right)
for r in ("bm25", "emb"):
    print(f"{r:5s} relevant chunk in context:", sum(any(c in q['relevant'] for c in c2[q['id']][r + '_hits']) for q in qa), "/6")


LLM only       agrees with reference: 0/6
BM25 RAG       agrees with reference: 6/6
Embedding RAG  agrees with reference: 6/6
bm25  relevant chunk in context: 6 /6
emb   relevant chunk in context: 6 /6


## C3 — question 1 of 3: C-03 (minimum GPA — the archived-rule trap)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (15.799), **GO-02 (14.059)** ✓, GO-06 (13.852) — the right chunk is only second; first is "Combining the grant with other support".

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> **GO-02 (0.629)** ✓, GO-20 (0.569), **GO-05 (0.545)** — the right chunk first, and the archived 2025 rule third.

**3. What was sent to the model** (the context blocks):

> BM25 RAG: [1] Combining the grant with other support (GO-16) · [2] Who qualifies (GO-02: "cumulative GPA is at least 2.67 on the 4.0 scale") · [3] Form GR-104 (GO-06). Embedding RAG: [1] Who qualifies (GO-02) · [2] Who cannot apply (GO-20) · [3] Archived: the 2025 scheme (GO-05: "the minimum GPA for the grant was 3.0 … do not apply to any application made for the 2026 academic year"). The full blocks are printed in the cell below.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> **More accurate and grounded.** Without context the model would not give a number at all ("I cannot provide a specific GPA number"). With context both systems answered 2.67 and cited the block that says it ([2] for BM25, [1] for embeddings). The embedding context contained the trap — GO-05's 3.0 — and the answer became *more specific* rather than wrong: "a previous archived 2025 scheme required … 3.0, but those rules were replaced on 1 June 2026".

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed in C2, but both failure modes were one step away, and the Top-K experiment shows them. **Retrieval:** BM25's rank 1 is the wrong chunk; at K = 1 BM25 RAG sends only GO-16 and answers "I do not know, as the provided context does not specify the minimum GPA" — a retrieval failure (the fact was not in the context; generation did the right thing with what it got). With Q-04's wording ("the minimum GPA for the grant") BM25 even ranks the archived GO-05 first. **Generation:** the embedding context put 3.0 and 2.67 side by side; the model chose correctly only because GO-05's own text says "ARCHIVED … no longer in force". The safeguard was in the document, not in the pipeline.

In [11]:
def take_apart(qid):
    q = next(x for x in qa if x["id"] == qid)
    print(f"{qid}: {q['question']}\nreference: {q['reference']}   relevant: {q['relevant']}\n")
    for r, name in (("bm25", "BM25"), ("emb", "EmbeddingGemma")):
        hits = RETRIEVERS[r](q["question"], 3)
        print(f"{name} top 3: " + ", ".join(f"{cid} ({s:.3f}){' *' if cid in q['relevant'] else ''}" for cid, s in hits))
    for r, name in (("bm25", "BM25 RAG"), ("emb", "Embedding RAG")):
        hits, out = rag(q["question"], r)
        print(f"\n----- context blocks sent by {name} -----")
        print(build_context(hits))
        print(f"----- {name} answer ({out['prompt_tokens']} prompt tokens) -----")
        print(out["answer"])
    print("\n----- LLM only (no context) -----")
    print(llm_only(q["question"])["answer"][:700], "…")
    print("\n(* = chunk listed in `relevant`)")

take_apart("C-03")

C-03: What is the minimum GPA needed to qualify for the need-based grant?
reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)   relevant: ['GO-02']

BM25 top 3: GO-16 (15.799), GO-02 (14.059) *, GO-06 (13.852)


EmbeddingGemma top 3: GO-02 (0.629) *, GO-20 (0.569), GO-05 (0.545)

----- context blocks sent by BM25 RAG -----
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[3] Form GR-104: the applicati

## C3 — question 2 of 3: C-04 (the alphabet switch — a RAG answer worse than I expected)

**1. What BM25 retrieved** (top 3, ids and scores):

> **KZ-43 (25.966)** ✓, KZ-14 (24.062), KZ-06 (16.835).

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> KZ-14 (0.720), **KZ-43 (0.693)** ✓, KZ-05 (0.434).

**3. What was sent to the model** (the context blocks):

> Both contexts contain the same two facts, in a different order. KZ-43 (Kazakh alphabets): "In October 2017, a presidential decree … ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031." KZ-14 (Kazakh language, part 3): "In October 2017 … Nazarbayev decreed … by 2025 … It is scheduled to be phased in from 2023 to 2031." Third block: KZ-06 Almaty (BM25) / KZ-05 Astana (embeddings), both irrelevant.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> **More accurate**: without context the model said the switch was "announced by President Tokayev in September 2018" and that the deadline "remains 2025" — wrong on the year, the person and the deadline, although this is Wikipedia text it has seen. Embedding RAG gave the full reference answer (October 2017; 2025 pushed back to 2031 in January 2021). BM25 RAG got both asked facts (October 2017, 2031) but wrote "scheduled to be completed by **2023 to 2031**" and dropped the move from 2025.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> BM25 RAG missed that the target was **moved** (2025 → 2031, January 2021) and turned KZ-14's phase-in window into a completion date. This is a **generation problem**: the fact was in its context — KZ-43 was its block [1], and it says "In January 2021, the target year for finishing the transition was pushed back to 2031". The model merged two chunks that describe the same plan differently. The embedding system got the same two chunks (KZ-14 first) and answered correctly — the same facts, in another order, gave a different answer. I still marked it ✓, because both things the question asks are in it.

In [12]:
take_apart("C-04")

C-04: When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
reference: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.   relevant: ['KZ-43']

BM25 top 3: KZ-43 (25.966) *, KZ-14 (24.062), KZ-06 (16.835)


EmbeddingGemma top 3: KZ-14 (0.720), KZ-43 (0.693) *, KZ-05 (0.434)



----- context blocks sent by BM25 RAG -----
[1] Kazakh alphabets
The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia. In October 2017, a presidential decree in Kazakhstan ordered a transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the target year for finishing the transition was pushed back to 2031.

[2] Kazakh language (part 3)
In October 2017, the then president, Nursultan Nazarbayev, decreed that the writing system would change from Cyrillic to Latin script by 2025. The proposed Latin alphabet has been revised several times and, as of January 2021, is close to the inventory of the Turkish alphabet, though it lacks the letters C and Ç and has four additional letters: Ä, Ñ, Q, and Ū (though other lett


----- context blocks sent by Embedding RAG -----
[1] Kazakh language (part 3)
In October 2017, the then president, Nursultan Nazarbayev, decreed that the writing system would change from Cyrillic to Latin script by 2025. The proposed Latin alphabet has been revised several times and, as of January 2021, is close to the inventory of the Turkish alphabet, though it lacks the letters C and Ç and has four additional letters: Ä, Ñ, Q, and Ū (though other letters, such as Y, have different values in the two languages). It is scheduled to be phased in from 2023 to 2031. Over one million Kazakh speakers in Xinjiang use a modified version of the Perso-Arabic script for writing.

[2] Kazakh alphabets
The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic script is used in Iran, Afghanistan, and China, while the Cyrillic script is used in Kazakhstan, Kyrgyzstan, Russia, and Mongolia

## C3 — question 3 of 3: C-06 (master's students — no answer in the corpus)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (16.996), GO-01 (14.173), **GO-20 (13.720)** ✓.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> **GO-20 (0.666)** ✓, GO-16 (0.616), GO-01 (0.554).

**3. What was sent to the model** (the context blocks):

> The same three chunks to both systems, in a different order: GO-16 (combining support; "capped at 400,000 tenge"), GO-01 (about the grant; "for full-time undergraduate students"), GO-20 (who cannot apply: "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe").

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> **More grounded — the only correct answer is "I do not know", and both RAG systems gave it, with the reason.** BM25 RAG: "Based on the provided context, I do not know … 'Master's and doctoral students are supported by a separate scheme, which this handbook does not describe' [3]". Embedding RAG: "I do not know … [1]". The LLM-only answer was a confident page on US graduate funding ("TA/RA Stipends: $20,000–$40,000/year"), useless for this office.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing — but this is where a generation failure would be most tempting: the context held amounts ("capped at 400,000 tenge") and the model could have answered "up to 400,000". It followed the system instruction instead. Had GO-20 not been retrieved, "I do not know" would still have been the right output but for the wrong reason (absence of evidence rather than the handbook's explicit exclusion) — that would be a retrieval failure hidden by a correct answer. Both retrievers found GO-20 (BM25 only in third place), so retrieval did not fail here.

In [13]:
take_apart("C-06")

C-06: I am a master's student. How much need-based grant can I get?
reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.   relevant: ['GO-20']

BM25 top 3: GO-16 (16.996), GO-01 (14.173), GO-20 (13.720) *


EmbeddingGemma top 3: GO-20 (0.666) *, GO-16 (0.616), GO-01 (0.554)

----- context blocks sent by BM25 RAG -----
[1] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] About the need-based study grant 2026
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without taking on paid work that harms their studies. One grant is awarded per student per academic year. The grant is a gift, not a loan: it does not have to be paid back unless the student withdraws early, as explained in the section on academic progress.

[3] Who cannot apply
The


----- context blocks sent by Embedding RAG -----
[1] Who cannot apply
The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules.

[2] Combining the grant with other support
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[3] About the need-based study grant 2026
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income household can pay for study costs without

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 scores **shared tokens**: each query word that appears in a chunk adds its idf (rarer = more),
> saturated by term frequency and normalised by chunk length. An embedding model turns the query and every
> chunk into a dense vector of meaning (768 numbers here) and ranks by cosine similarity, so wording and
> even language matter much less. Part A shows it in miniature: as written, BM25 gave all three documents
> 0 because *RAG* ≠ *Retrieval-Augmented Generation*; the vectors still ranked D2 first.

**2. When can BM25 perform better than embeddings?**

> When the query hinges on an exact, rare string — a number, a code, a name — whose meaning the embedding
> cannot place. In my runs: **Q-13 "456"** — BM25 found the only chunk containing `456` (KZ-38); EmbeddingGemma
> returned three unrelated form/date chunks at cosine < 0.2. BM25 is also transparent (I can list which
> word scored, as for Q-04's *minimum* +4.57), needs no model, and indexes 74 chunks instantly.

**3. When can embeddings perform better than BM25?**

> When the right chunk uses different words or a different language: paraphrase (**Q-02** *poor* ↔
> *low-income household*), cross-language (**Q-07** English question, Russian GO-15; **Q-10** Russian
> question, English GO-02), and when the query's exact words also fit a wrong chunk (**Q-04**: BM25 put the
> archived GO-05 first). Over the ten required queries embeddings hit 10/10, BM25 6/10.

**4. Why is cosine similarity used with embeddings?**

> Because in an embedding space the **direction** carries the meaning, and the length mostly reflects
> things like text length or how frequent the words are. Cosine compares only the angle, so a long chunk
> is not favoured just for being long, and the score lives on a fixed −1…1 scale that is comparable
> between queries. EmbeddingGemma is trained so that related texts point the same way. In Part A, D2's
> vector is the longest (‖D2‖ = 1.12 vs 0.81 for D1), yet its cosine is only 0.002 higher than D1's.

**5. What information is actually sent to an LLM in a RAG system?**

> Text, and only text: the system message, then a user message with "Context:", the retrieved chunks as
> numbered blocks (`[n] title` + chunk text) and "Question: …". In C1 that was 44 + 302 + 18 + 4 = 368
> tiktoken tokens (415 by the provider's count, with the chat template). No document, link, index or
> score is sent unless my code writes it into the prompt.

**6. Does the LLM search the original documents itself?**

> No. It never sees the corpus; it continues the prompt my code built. Evidence: KZ-44 (Narxoz University)
> is in the corpus, yet the LLM-only answer to C-05 invented a founding in 2013 as the "International
> University of Tashkent"; with KZ-44 pasted into the prompt, both RAG systems answered 1963 and the
> Almaty Institute of National Economy. The search is done by BM25 or EmbeddingGemma, in my code, before
> the call.

**7. What happens if retrieval returns an irrelevant chunk?**

> It costs tokens and may mislead. If a relevant chunk is also there, the model usually ignores the
> irrelevant one: BM25's rank 1 for C-03 was GO-16 (support caps), and the answer still came from block [2]
> (GO-02). If the irrelevant chunk is the *only* thing sent, the model says it does not know (BM25 at K = 1
> on C-03) — or, worse, answers from a plausible distractor: the archived GO-05 (3.0) would give a
> confidently wrong answer if its text did not say "ARCHIVED".

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> On **C-03** (minimum GPA), from the cell below:
>
> | Retriever | K = 1 | K = 3 | K = 8 |
> |---|---|---|---|
> | BM25 | context [GO-16] → **"I do not know"** (169 prompt tokens) | [GO-16, GO-02, GO-06] → **2.67** ✓ (386) | 8 chunks incl. archived GO-05 → 2.67 ✓ + "older 2025 scheme required 3.0 … no longer in force" (934) |
> | Embeddings | [GO-02] → **2.67** ✓ (194) | [GO-02, GO-20, GO-05] → 2.67 ✓ + archived note (380) | 8 chunks → 2.67 ✓ + archived note (876) |
>
> A larger K **improves recall**: BM25's right chunk was only second, so K = 1 failed and K = 3 fixed it.
> But it also **adds distractors and cost**: K = 8 sends 2.3–2.4× the tokens of K = 3 for the same answer, and
> brings in the archived 3.0 rule for both retrievers, so the right answer now depends on the model
> resolving a conflict. Here it did; with a less clearly labelled distractor, or a fact buried among
> eight blocks, a bigger K makes the answer worse, not better.

In [14]:
# Top-K experiment on C-03 (minimum GPA): what changes with K = 1, 3 and 8, for both retrievers.
q = next(x for x in qa if x["id"] == "C-03")
for r in ("bm25", "emb"):
    for k in (1, 3, 8):
        hits, out = rag(q["question"], r, k)
        ctx_tokens = len(enc.encode(build_context(hits)))
        print(f"[{r} K={k}] context {[cid for cid, _ in hits]}  ({ctx_tokens} context tokens, "
              f"{out['prompt_tokens']} prompt tokens)")
        print("   ", out["answer"].replace("\n", " ")[:400])
    print()

[bm25 K=1] context ['GO-16']  (81 context tokens, 169 prompt tokens)
    I do not know, as the provided context does not specify the minimum GPA required to qualify for the need-based grant. The context only mentions that the grant is need-based and cannot be combined with other need-based grants, but it does not list academic performance requirements like GPA). [1]
[bm25 K=3] context ['GO-16', 'GO-02', 'GO-06']  (284 context tokens, 386 prompt tokens)
    The minimum GPA needed to qualify for the need-based grant is 2.67 on the 4.0 scale [2].
[bm25 K=8] context ['GO-16', 'GO-02', 'GO-06', 'GO-05', 'GO-01', 'GO-11', 'GO-03', 'GO-17']  (788 context tokens, 934 prompt tokens)
    The minimum GPA needed to qualify for the need-based grant is 2.67 on the 4.0 scale [2].  Note: While an older 2025 scheme required a GPA of 3.0, those rules are no longer in force and do not apply to the 2026 academic year [4].



[emb K=1] context ['GO-02']  (108 context tokens, 194 prompt tokens)
    Based on the provided context, the minimum GPA needed to qualify for the 2026 grant is **2.67** on the 4.0 scale [1].


[emb K=3] context ['GO-02', 'GO-20', 'GO-05']  (274 context tokens, 380 prompt tokens)
    The minimum GPA needed to qualify for the 2026 need-based grant is 2.67 on the 4.0 scale. [1]  Note that a previous archived 2025 scheme required a higher minimum GPA of 3.0, but those rules were replaced on 1 June 2026 and do not apply to the 2026 academic year. [3]
[emb K=8] context ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']  (744 context tokens, 876 prompt tokens)
    The minimum cumulative GPA required to qualify for the 2026 need-based study grant is 2.67 on a 4.0 scale [1].  Note: The 2025 scheme had a higher minimum GPA of 3.0, but those rules are no longer in force [3].



**9. Is a vector database required to build a RAG system?**

> No. My "vector store" is one NumPy array, `doc_emb`, of shape (74, 768), float32 — about 227 KB in
> memory — and retrieval is a brute-force cosine against all 74 rows (`model.similarity`), which takes
> milliseconds. BM25 needs no vectors at all. A vector database becomes useful with millions of chunks
> (approximate-nearest-neighbour indexes), persistence, updates and metadata filters — for example, to
> exclude chunks marked *archived*, which this corpus would need.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> **EmbeddingGemma.** hit@3 over the ten queries Q-01–Q-10: **embeddings 10/10, BM25 6/10** (over all 13
> including my extra queries: 12/13 vs 9/13). BM25 missed the paraphrase (Q-02), the archived-rule trap
> (Q-04) and both cross-language queries (Q-07, Q-10) — and this corpus deliberately has Russian and
> Kazakh chunks and an archived rule. In C2 both RAG systems agreed with the reference **6/6** (LLM only:
> **0/6**), and the relevant chunk was in the top 3 for 6/6 questions with both retrievers: at K = 3 the
> model absorbed BM25's noisier ranking. The difference showed at K = 1 (BM25 failed C-03) and in answer
> quality (C-04, where the embedding context produced the complete answer). BM25 won only on a bare rare
> number (Q-13).

---
# Conclusion

> RAG here was a search problem first: once the right chunk was in the prompt, the model answered every
> question correctly and said "I do not know" when it should, and without retrieval it answered none —
> including Wikipedia facts it has seen. What I would build next is **hybrid retrieval**: BM25 and
> EmbeddingGemma fused by reciprocal rank, because they fail on different queries (BM25 on paraphrase
> and other languages, embeddings on bare numbers and codes like Q-13). I would add a **metadata filter**
> so the archived GO-05 can never be retrieved for a 2026 question instead of relying on the word
> "ARCHIVED" inside the text, keep these 13 queries with their `relevant` ids as a regression set
> (hit@3 per retriever, rerun on every change), and check citations in code — each `[n]` must point to a
> block that contains the number the answer states.